# MCP Lab: Туристический советник с погодным MCP-сервером

В этой лабораторной работе мы создадим **AI-агента — туристического советника**, который подбирает место для путешествия на основе предпочтений пользователя и **актуальной погоды** в городах-кандидатах.

Агент:
1. Ищет в интернете подходящие направления (Web Search)
2. Запрашивает текущую погоду через **MCP-сервер** (OpenWeatherMap)
3. Формирует таблицу с рекомендациями

## Архитектура

```
┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐
│  Jupyter Client │──────│   Yandex Cloud  │──────│   Yandex        │
│  (этот ноутбук) │      │   Responses API │      │   Search        │
└─────────────────┘      │                 │      └─────────────────┘
                         │                 │      ┌─────────────────┐
                         │                 │──────│    MCP Hub      │
                         └─────────────────┘      │  (Yandex Cloud) │
                                                  └────────┬────────┘
                                                           │
                                                  ┌────────▼────────┐
                                                  │ OpenWeatherMap  │
                                                  │ service (REST)  │
                                                  └─────────────────┘
```

**Ключевые особенности**:
* Responses API сам подключается к MCP-серверу и вызывает нужные функции!
* Для создания MCP-сервера из общедоступного OpenWeatherMap API мы используем MCP Hub Yandex Cloud - это получится сделать без единой строчки кода.

## План работы

| Часть | Описание |
|-------|----------|
| 1 | Настройка окружения и авторизация |
| 2 | Создание MCP-сервера для погоды |
| 3 | Настройка инструментов: Web Search + MCP |
| 4 | Создание агента-советника |
| 5 | Тестирование на разных запросах |

## Настройка окружения

Установим необходимые библиотеки:

In [ ]:
%pip install --quiet openai python-dotenv

> **ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

## Авторизация

Для работы с Yandex Cloud нам понадобится `folder_id` (идентификатор каталога) и `api_key` (API-ключ сервисного аккаунта). Мы предполагаем, что эти значения хранятся в файле `.env`, который можно скачать, выполнив ячейку ниже:

In [ ]:
!curl -o .env {{url_of_dotenv_file}}

Также скачаем реализацию класса `Agent` из репозитория. Этот класс универсально поддерживает все типы инструментов: словарные описания (Web Search, File Search, MCP), а также Pydantic-классы для локальных функций.

In [ ]:
!curl -o Agent.py https://raw.githubusercontent.com/yandex-ai-studio/ai-studio-course/refs/heads/main/4-rag-search/Agent.py

Загрузим переменные окружения и создадим клиент для работы с API:

In [ ]:
import os, json
from dotenv import load_dotenv
from IPython.display import Markdown, display
from Agent import Agent, create_client

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]
model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = create_client()

def printx(string):
    display(Markdown(string))

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")

---

## Часть 1: Создание MCP-сервера для погоды

Прежде чем создавать агента, нам необходимо настроить **MCP-сервер**, который будет предоставлять данные о погоде через [OpenWeatherMap API](https://openweathermap.org/).

### Что такое OpenWeatherMap?

[OpenWeatherMap](https://openweathermap.org) - это общедоступный сервис, предоставляющий через REST API данные о погоде на текущий момент, а также исторические данные и предсказания.

Мы, для простоты, будем использовать текущие данные о погоде - функцию [CurrentWeatherData](https://old.openweathermap.org/current). Она позволяет получить данные о погоде с помощью простого REST-запроса:

```
https://api.openweathermap.org/data/2.5/weather?lat=44.34&lon=10.99&appid={API key}
```

> Получить свой ключ **API key** для использования сервиса можно после бесплатной регистрации.

В примере выше нам потребуются широта и долгота города, которые можно получить отдельно с помощью ещё одного вызова [GeoCoder API](https://old.openweathermap.org/api/geocoding-api). Но есть и небольшой лайфхак - можно сразу передать в запрос название города вместо широты и долготы (правда, такой формат вызова является устаревшим и может перестать поддерживаться):

```
https://api.openweathermap.org/data/2.5/weather?q={City_Name}&appid={API key}
```

> Рекомендуется также добавить параметр units=metric, чтобы результат возвращался в градусах Цельсия 

### Два способа создания MCP-сервера

Нам необходимо "обернуть" REST API OpenWeatherMap в формат MCP, чтобы использовать его в нашем агенте. Это можно сделать, написав специальный шлюз на Python и запустив его работать в облаке на виртуальной машине, однако это требует постоянных вычислительных ресурсов. Можно более эффективно реализовать такую обертку с помощью MCP Hub в Yandex Cloud, одним из двух способов:

#### Вариант 1: REST-эндпоинт (прямой API)

Самый простой способ — подключить REST API OpenWeatherMap напрямую через MCP Hub, который позволяет обернуть любое REST API в формат MCP.

**Преимущества** | **Ограничения**
----|----
Простота настройки — не нужна дополнительная инфраструктура | Нет возможности добавить логику (например, геокодирование)
Нет дополнительных затрат | Формат ответа фиксирован


📹 [Видеоинструкция: настройка REST MCP-эндпоинта](https://youtu.be/EiZtWfhr-z4)

#### Вариант 2: Cloud Functions

Более гибкий вариант — создать Cloud Function, которая выполняет дополнительную логику перед запросом к OpenWeatherMap. Например, мы можем следовать рекомендациям и сначала вызвать **GeoCoder API** для получения координат города по имени, а затем — сделать запрос погоды по точным координатам. Пример реализации такой облачной функции содержится [тут](cloud-getweather/index.py)


**Преимущества:** | **Ограничения:**
----|----
Можно добавить произвольную логику (геокодирование, валидацию, форматирование) | Немного сложнее в настройке 
Оплата только за вызовы — не нужно заботиться об инфраструктуре | Требует программирования
Удобно масштабировать | Требует размещения в Yandex Cloud

📹 [Видеоинструкция: настройка Cloud Functions MCP-эндпоинта](https://youtu.be/7L_MU66VvSo)

### Используем готовый MCP-сервер

Для данной лабораторной вам предлагается настроить погодный MCP-сервер одним из приведённых выше способов (а лучше - обоими). Ниже вставьте URL MCP-сервера:

In [ ]:
mcp_server_url = "--вставьте адрес MCP-сервера--"


**Задание:** Определите MCP-инструмент для работы с погодным сервером.

In [ ]:
# Опишите MCP-инструмент для погоды

Давайте проверим, что MCP-сервер работает, выполнив простой запрос через Responses API. На данном этапе мы не используем класс `Agent` — просто отправляем запрос напрямую, чтобы убедиться в корректности подключения:

In [ ]:
# Быстрая проверка MCP-сервера
# Используйте Responses API, чтобы проверить работу MCP-сервера

Посмотрим на структуру ответа, чтобы понять, как MCP-вызовы отображаются в результатах:

In [ ]:
# Проанализируйте ответ и убедитесь, что MCP-вызов был сделан

---

## Часть 2: Настройка инструмента Web Search

Помимо MCP-сервера для погоды, нашему агенту нужен инструмент **Web Search** для поиска туристических направлений. Агент будет искать в интернете подходящие места на основе запроса пользователя.

**Задание:** Настройте инструмент Web Search.

In [ ]:
# Инструмент для поиска в интернете
# Опишите словарь для задания инструмента поиска

---

## Часть 3: Создание агента — туристического советника

Теперь у нас есть оба инструмента:
- **Web Search** — для поиска направлений в интернете
- **Weather MCP** — для получения актуальной погоды

Создадим агента, который комбинирует оба инструмента. Ключевая особенность — **системный промпт**, который задаёт стратегию работы агента:

1. Понять запрос пользователя (какая погода/атмосфера желательна)
2. Через Web Search найти 3–5 подходящих городов
3. Через MCP Weather запросить текущую погоду в каждом городе
4. Составить таблицу с top-3 рекомендациями

**Задание:** Создайте агента с двумя инструментами и подробным промптом.

In [ ]:
travel_instruction = """
... опишите системный промпт ...
"""

travel_agent = Agent(
    client=client,
    instruction=travel_instruction,
    tools=[web_search_tool, weather_mcp_tool],
    tool_choice="auto"
)

---

## Часть 4: Тестирование агента

Давайте проверим работу агента на нескольких запросах разной степени конкретности. Обратите внимание, как агент использует оба инструмента: поиск в интернете для нахождения кандидатов и MCP-сервер для уточнения текущей погоды.

### Тест 1: Поиск тёплого места

**Задание:** Попросите агента найти самое тёплое место для отдыха.

In [ ]:
result = travel_agent("Хочу поехать в самое тёплое место! Где сейчас можно погреться на солнце?")
printx(result.output_text)

### Тест 2: Экзотический запрос — дождь и меланхолия

Попробуем нестандартный запрос — агент должен понять эмоциональный контекст и подобрать подходящие направления.

In [ ]:
travel_agent.reset()
result = travel_agent(
    "Хочу полностью окунуться в дождь, почувствовать меланхолию и депрессию. "
    "Где сейчас самые дождливые и серые места?"
)
printx(result.output_text)

### Тест 3: Практичный запрос — снег для горнолыжного отдыха

Ещё один вариант — запрос с конкретной целью.

In [ ]:
travel_agent.reset()
result = travel_agent(
    "Ищу лучшее место для горнолыжного отдыха прямо сейчас. "
    "Где сейчас хороший снег и не слишком холодно?"
)
printx(result.output_text)

### Тест 4: Уточняющий диалог

Одно из преимуществ класса `Agent` — сохранение контекста между сообщениями. Можно вести диалог, уточняя предпочтения:

In [ ]:
travel_agent.reset()
result = travel_agent("Хочу куда-нибудь в Азию, где сейчас комфортная погода для прогулок.")
printx(result.output_text)

In [ ]:
# Уточняющий вопрос — контекст сохраняется
result = travel_agent("А какой из этих городов самый дешёвый для проживания?")
printx(result.output_text)

---

## Как это работает — под капотом

Давайте разберёмся, что происходит «под капотом», когда агент обрабатывает запрос.

```
Пользователь: "Хочу в самое тёплое место!"
     │
     ▼
┌─────────────────────────────────────────────┐
│ LLM анализирует запрос и решает использовать│
│ Web Search для поиска тёплых направлений    │
└─────────────────────────────────────────────┘
     │
     ▼ Web Search → результаты: Дубай, Бангкок, Канкун...
     │
     ▼
┌─────────────────────────────────────────────┐
│ LLM видит кандидатов и решает запросить     │
│ погоду через MCP Weather для каждого        │
└─────────────────────────────────────────────┘
     │
     ▼ MCP Weather("Dubai") → 35°C, sunny
     ▼ MCP Weather("Bangkok") → 32°C, cloudy
     ▼ MCP Weather("Cancún") → 28°C, partly cloudy
     │
     ▼
┌─────────────────────────────────────────────┐
│ LLM формирует финальную таблицу с top-3     │
│ рекомендациями на основе всех данных        │
└─────────────────────────────────────────────┘
```


---

## Заключение

В этой лабораторной работе мы:

1. **Познакомились с MCP** — протоколом для подключения LLM к внешним источникам данных
2. **Рассмотрели два способа создания MCP-сервера** — через REST API и через Cloud Functions
3. **Настроили погодный MCP-сервер** на базе OpenWeatherMap
4. **Создали агента с двумя инструментами** — Web Search + MCP Weather
5. **Протестировали агента** на разнообразных запросах — от "хочу тепла" до "хочу дождя и депрессии"

### Ключевые выводы

- **MCP — мощный механизм расширения** возможностей LLM через стандартизированный протокол
- **Responses API упрощает интеграцию** — не нужно писать MCP-клиент
- **Комбинирование инструментов** (Web Search + MCP + Pydantic) позволяет создавать сложных агентов
- **Системный промпт** играет ключевую роль в качестве ответов агента — чем подробнее стратегия, тем лучше результат

### Идеи для расширения

- Добавить поиск авиабилетов через отдельный MCP-сервер
- Подключить MCP-сервер для конвертации валют
- Создать Pydantic-инструмент для сохранения маршрутов в файл
- Добавить MCP-сервер с информацией о визовых требованиях